# 00 · How it works

This notebook is the long-form explanation of the method and of the code. It is mostly text:
three small code cells import the package and print some of its constants, but nothing is
fitted, run or downloaded, and no API key is needed.

- **Short version:** [`FLOW.md`](../FLOW.md) has one round, the design criteria, the protocol,
  the run folder and a compact glossary on one page.
- **Running things:** [`USAGE.md`](../USAGE.md). **What the code needs:**
  [`REQUIREMENTS.md`](../REQUIREMENTS.md).
- **Seeing it work:** notebooks [01](01_engine_basics.ipynb) to
  [04](04_reading_a_run_folder.ipynb) run the engine with the mock proposer and reproduce its
  recorded numbers.
- **Needs:** the base install. **Runtime:** a few seconds.

Contents:

1. Setup
2. The loop in brief
3. Notation
4. Round 0
5. One round, step by step
6. Validation, model identity and the prior probe
7. Fitting and the evidence
8. Evidence weights, and why they are conditional on the pool
9. Prediction and scores
10. The design criteria
11. What never reaches the model
12. The protocol of the paper runs
13. The benchmarks
14. Glossary

## 1. Setup

The cell finds the repository root, makes the package importable and prints the versions of the
numerical stack. [`REQUIREMENTS.md`](../REQUIREMENTS.md) lists the pinned versions.

In [1]:
import hashlib
import logging
import os
import sys
from importlib.metadata import version
from pathlib import Path


def find_repo_root(start):
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "bayes_model_discovery").is_dir():
            return folder
    raise FileNotFoundError("start Jupyter inside the repository checkout")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
try:
    import bayes_model_discovery
except ImportError:  # not installed: use the source tree directly
    sys.path.insert(0, str(REPO / "src"))
    import bayes_model_discovery
logging.getLogger("pytensor.tensor.blas").setLevel(logging.ERROR)

print(f"{'bayes_model_discovery':24}{bayes_model_discovery.__version__}")
for name in ("numpy", "scipy", "pymc", "pytensor", "anthropic"):
    print(f"{name:24}{version(name)}")

bayes_model_discovery   0.1.0.dev0
numpy                   1.26.4
scipy                   1.17.1
pymc                    5.13.1
pytensor                2.20.0
anthropic               0.120.2


## 2. The loop in brief

A run studies one **world**: a process with a hidden true law that answers experiments. In each
**round** the engine

1. asks a language model, the **proposer**, for candidate models, given a **belief packet** that
   summarizes the public data and the current pool;
2. validates each candidate and fits it by **SMC** (sequential Monte Carlo) from its prior on
   all public data;
3. keeps the twelve models with the highest marginal likelihood (**evidence**) as the **pool**;
4. predicts by averaging the pool's predictions with **evidence weights** (Bayesian model
   averaging, **BMA**);
5. chooses the next experiment, the **design**, by a **design criterion**, observes its outcome
   and adds it to the public data.

The loop runs for a fixed number of rounds. Held-out data score the predictions after each round
but never influence the run. In place of a language model, the **mock proposer** answers from a
fixed script; it costs nothing and is the default everywhere. Section 5 gives the exact order of
the steps in the code.

## 3. Notation

- $D$: the public data, that is, the initial design $D_0$ and every observation made since.
- $x$: a design (the inputs of one experiment); $y$: an outcome.
- $m$: a model, meaning an expression, a likelihood family and the roles of its coefficients;
  $\theta$: its coefficients.
- $\mathcal{M}$: the pool, at most twelve models (six in the NeuronBench paper profile).
- $Z_m = p(D \mid m)$: the evidence of model $m$; $w_m$: its evidence weight.
- $\theta_{m,k}$, $k = 1, \dots, K_m$: the posterior particles of model $m$ used in a calculation.
- $\bar p_m(y \mid x) = \frac{1}{K_m} \sum_k p(y \mid x, m, \theta_{m,k})$: model $m$'s
  predictive distribution.
- $\bar p(y \mid x) = \sum_m w_m \, \bar p_m(y \mid x)$: the pool's predictive distribution, the
  BMA.
- $H[p] = -\sum_y p(y) \log p(y)$: entropy in nats.

## 4. Round 0

1. **Observe the initial design $D_0$**, at designs fixed by a rule rather than chosen by a
   criterion. The synthetic count runner observes the doses 0.1, 0.35, 0.65 and 0.9
   (`--d0-policy fixed4`; `stratified4` draws one dose in each quarter of [0, 1] from a
   generator seeded by the world). BoxingGym offers `fractions4` (the default, the task's own
   rule) and `random2` (two designs drawn from the candidate set with a separate random stream;
   three for the `location` task). ChemBench offers `random4` (the default) and `fixed3`. The
   ForceBench launch script starts from two random designs.
2. **Build the starting pool**, in one of four ways:
   - a set of starting structures: in the synthetic count benchmark `b0`, `b0+b1*z`,
     `b0+b1*z+b2*square(z)` and `b0+b1*log(1+u)`, with $z = 2u - 1$, in the family given by
     `--initial-family` (default `binomial`);
   - a fixed catalogue of starting models (ChemBench: four regression models,
     `--initial-catalogue on|off`);
   - a pool read from a file made once per task and seed (`--initial-pool-file`), so that every
     arm starts from the same models and round 0 makes no LLM call;
   - the **initial batch** of LLM proposals (`--initial-proposals K`): K parallel requests for
     one candidate each, then at most two top-up waves that ask only for the number still
     missing. Records: `proposals/t00_burst.json` and `packets/t00_burst_w<wave>.json`.

   Proposed candidates go through the same validation, prior probe and fitting as in later
   rounds.
3. **Admit and score.** The twelve models with the highest evidence are kept
   (`--active-maximum`, default 12) and the pool is scored once (phase `initial`).

## 5. One round, step by step

Rounds 1, 2, … each add exactly one observation. The steps follow the synthetic count runner
(`core_count.py`); the other runners go through the same steps in the same order (choose,
observe, refit, propose, validate, fit, admit) but differ in when they score. Module names refer
to `src/bayes_model_discovery/experimental/`.

1. **Choose.** The design criterion is evaluated on the candidate designs with the pool of the
   previous round, and the best design is taken. Ties within $10^{-12}$ of the maximum go to the
   smallest design; one function takes this argmax for every criterion
   (`eig.joint.choose_design`). Code: `design_policies.select`; `count_inference.choose` (joint
   expected information gain, EIG, and its parts) and `count_inference.choose_epig` (expected predictive
   information gain, EPIG) for bounded counts;
   `positive_inference.joint_eig` and `vector_inference.joint_eig` for other outcome types;
   `design_search.projected_spsa` for continuous search.
2. **Observe.** The world returns the outcome at the chosen design, and it joins $D$. Before
   refitting, each pool member's previous fit gives a one-step-ahead predictive log score for
   the new observation; the packet shows this history. Code: `core_count.CoreWorld.observe`,
   the ChemBench observation adapter, the BoxingGym environment's `step`, the neuronbench and
   DiscoverPhysics simulators.
3. **Refit.** Every pool member is refit by SMC from its prior on all of $D$, optionally under a
   time limit (`--fit-timeout`). Code: `count_inference.fit` (bounded counts),
   `positive_inference.fit`, `vector_inference.fit`; the time limit in
   `count_inference.run_with_fit_timeout`.
4. **Score.** The BMA predictive is scored on the held-out set (phase `before_proposal`). Code:
   `count_inference.predict`, `core_count.score_prediction`.
5. **Build the packet.** The public state is serialized as a belief packet: compact JSON with
   sorted keys and every number rounded to four significant digits, so that a packet is a
   stable, hashable record. Code: `belief_packet.build_belief_packet`,
   `belief_packet.serialize_packet`.
6. **Propose.** One request asks the LLM for k candidates (`--proposal-limit`, 1 to 3); on an
   archive review round the same request also invites up to two revive requests (section 12).
   The request is the system prompt, with paragraphs appended for multi-candidate and review
   requests, and the packet as the only user message. Code: `count_proposer.py`
   (`AnthropicCountProposer`, `OpenAICountProposer`, `OpenAICompatCountProposer`);
   `neuron_proposer.py` for NeuronBench. The mock proposer answers from a script instead.
7. **Validate.** Each candidate is checked (section 6). A reply that cannot be used gets one
   repair turn with the error message. Code: `count_models.validate_expression`,
   `count_models.judge_candidate`, `count_models.canonical_model_id`, `core_count.prior_probe`,
   `prior_probe_detail.first_nonfinite_subexpression`.
8. **Fit.** Each valid candidate, and each revived model, is fitted by SMC from its prior on all
   of $D$.
9. **Admit.** All models are sorted by log evidence (ties broken by model id) and the top twelve
   are kept; the rest are marked `retired_by_evidence` in the archive, with their log evidence,
   their gap to the cutoff in nats, the rounds they were active and the number of observations at
   the time. Code: `count_inference.retain`.
10. **Score again.** If the pool changed, it is scored again (phase `after_proposal`); otherwise
    the previous row is copied and marked `post_equals_pre: true`.

Differences between runners:

- ChemBench keeps each round's pool and scores all checkpoints against the benchmark's official
  evaluation set after the run has finished, because the true values are requested only then.
- NeuronBench's candidates are mechanism names rather than expressions, and its packet and
  proposer are its own (`neuron_proposer.py`).
- The ForceBench runner, `tools/fb1_snapshot/run_gravity.py`, follows the same steps with its
  own modules: `fb1_proposer.py`, `fb1_grammar.py`, `fb1_core.py`, `fb1_forward.py` and
  `fb1_design_search.py`.

## 6. Validation, model identity and the prior probe

**What a candidate is.** A candidate is a JSON object: an expression for the mean on the link
scale (for example the logit for counts out of a fixed number of trials and the identity for
real-valued outcomes), a likelihood family, and a role for each coefficient. The role fixes the
coefficient's prior. In the synthetic count runner the roles are `intercept` ~ Normal(0, 2),
`signed` ~ Normal(0, 1), `positive` ~ LogNormal(0, 1) and `shape` ~ Gamma(2, 1); the packet
lists them under `constraints`.

**Validation.** A candidate must

- parse under the grammar: arithmetic and the functions `exp`, `log`, `sqrt`, `tanh`, `square`
  and `power` (`--grammar-version v3` adds six more);
- use at most six distinct mean coefficients and fifteen operators;
- use only the roles and likelihood families the run allows (the condition code decides whether
  new structures and new families are allowed at all);
- not duplicate a model already in the pool, in the archive, or earlier in the same reply.

A reply that fails is answered once with the error message (one repair turn,
`--repair-attempts`). Validation events and rejection reasons are recorded in
`proposals/tNN.json` and in the archive.

**Model identity.** Two candidates are the same model when their `model_id` is equal. The id is
a canonical key computed from the expression's syntax tree, the coefficient roles and the
likelihood family (schema `ast_roles_family_ac_v2`, recorded as `model_id_schema`):

- renaming coefficients does not change it;
- swapping the two operands of one `+` or `*` does not change it (`b0+b1*z` and `b1*z+b0` share
  a key);
- a sum of three terms is two nested `+` operations, and terms are not moved from one to the
  other, so reordering a longer sum can give a different key (`b0+b2*square(z)+b1*z` differs
  from `b0+b1*z+b2*square(z)`);
- scaling a term changes it (`b0+b1*z` and `b0+2*b1*z` are different models, because their
  priors differ);
- `square(x)`, `power(x, 2)` and `x**2` get the same key.

A duplicate is recorded and not fitted again.

**Prior probe.** Before any candidate is fitted, the engine draws 64 coefficient vectors from the
candidate's fixed priors and evaluates the expression, the link and the log likelihood on a
16-point grid over the declared input range. If any value is non-finite, the candidate is
rejected without a fit, and the archive names the sub-expression, input value and range that
failed (`prior_probe_detail.py`). The probe's random generator is seeded from the model id
alone, so the probe consumes none of the run's random numbers. The same draws give the prior
predictive summary (mean and 5%, 50% and 95% quantiles at each grid point) stored in
`fits/*.json` (`core_count.prior_probe`).

## 7. Fitting and the evidence

Each model is fitted with sequential Monte Carlo (`pm.sample_smc` in PyMC). SMC moves a
population of particles from the prior to the posterior through a sequence of tempered
distributions and estimates the marginal likelihood on the way:

$$
Z_m = p(D \mid m) = \int p(D \mid m, \theta) \, p(\theta \mid m) \, d\theta .
$$

Each fit runs $C = 2$ chains. Each chain estimates the evidence, and the two estimates are
averaged on the evidence scale, not on the log scale:

$$
\hat Z_m = \frac{1}{C} \sum_{c=1}^{C} \hat Z_m^{(c)} .
$$

`fits/tNN_<model_id>.json` records the total and per-chain log evidence
(`per_chain_log_evidence`), the sampler diagnostics and the fit time; the `.npz` file beside it
holds the posterior particles.

Every fit starts from the model's prior and uses all public data. A model refit in a later round
is not updated from its earlier posterior, so its evidence is always the evidence of all data so
far.

**Profiles.** `--profile` selects a compute preset; all profiles use the same priors, grammar,
likelihoods and sampler.

| Runner | `dev` | `paper` |
| --- | --- | --- |
| synthetic count | 64 particles × 2 chains, 16 candidate designs, 32 particles per model for EIG, 32 held-out inputs | 500 particles × 2 chains, 100 candidate designs, 128 particles per model for EIG, 256 held-out inputs |
| ChemBench | 64 particles × 2 chains, 32 candidate designs, 32 EIG samples, 32 held-out inputs | 500 particles × 2 chains, 256 candidate designs, 128 EIG samples, 256 held-out inputs |

NeuronBench adds `paper_neuron`: 200 particles, two chains, three fixed Metropolis–Hastings
moves per stage, and the benchmark's six evaluation features. The default profile is `paper` in
the synthetic count runner and `dev` in the ChemBench, BoxingGym and NeuronBench runners.

## 8. Evidence weights, and why they are conditional on the pool

The weight of model $m$ is its share of the pool's total evidence,

$$
w_m = \frac{\hat Z_m}{\sum_{j \in \mathcal{M}} \hat Z_j} ,
$$

computed in log space (`count_inference.model_logweights`). This is Bayes' rule over the models
in the pool with a uniform prior on them, $p(m \mid \mathcal{M}) = 1 / |\mathcal{M}|$:

$$
p(m \mid D, \mathcal{M}) = \frac{p(D \mid m) \, p(m \mid \mathcal{M})}{\sum_{j \in \mathcal{M}} p(D \mid j) \, p(j \mid \mathcal{M})} = w_m .
$$

The condition on $\mathcal{M}$ matters:

- **The weights compare only the models in the pool.** Adding or retiring one model changes
  every weight. Two models with log evidence −20 and −22 have weights $1 / (1 + e^{-2}) \approx
  0.88$ and $0.12$. A third model with log evidence −19 takes about 0.71 and leaves about 0.26
  and 0.035 to the first two, although nothing about them changed.
- **A weight close to 1 is relative.** It says that a model is far better supported than the
  other pool members, not that it is the true law. A better model that was never proposed does
  not lower it.
- **Admission is decided by the evidence.** After the fits of a round, all models are ranked by
  log evidence and the top twelve stay. A retired model can come back only through an archive
  review (section 12); proposing it again counts as a duplicate.
- **The evidence uses only public data.** Held-out scores never decide admission or weights.

## 9. Prediction and scores

The pool predicts by Bayesian model averaging: first over each model's particles, then over
models with the evidence weights.

$$
\bar p(y \mid x) = \sum_{m \in \mathcal{M}} w_m \, \frac{1}{K_m} \sum_{k=1}^{K_m} p(y \mid x, m, \theta_{m,k})
$$

The runners also record a single representative model, the one with the highest evidence (ties
broken by model id). It is never chosen by its test score.

The BMA is scored on a held-out evaluation set that is never used for fitting, choosing designs
or proposing (in the synthetic count runner 32 inputs in `dev` and 256 in `paper`, with four
outcomes each):

- **NLL**: the negative log predictive probability (or density) of the held-out observations, in
  nats per observation. Lower is better.
- **coverage90, width90**: the fraction of held-out observations inside the 90% predictive
  interval, and the average width of that interval.
- **KL**: the KL divergence from the true outcome distribution to the BMA predictive,
  $\mathrm{KL}\big(p_{\text{true}}(\cdot \mid x) \,\|\, \bar p(\cdot \mid x)\big)$, averaged over
  the evaluation inputs.

Each scored state of the pool is a **checkpoint**; its scores, pool members and weights are
written to `scores.jsonl`, and the same checkpoints to `points.csv`.

## 10. The design criteria

In each round the next design is the candidate design with the highest value of the run's
criterion. The paper compares five criteria, the **arms**. Joint, Model and Parameter EIG are
computed from one set of posterior draws, so at every design the identity joint EIG = Model EIG
+ Parameter EIG holds exactly. It is the chain rule of mutual information:

$$
I\big((M, \theta); Y \mid x, D\big) = I(M; Y \mid x, D) + I(\theta; Y \mid M, x, D) .
$$

**Joint EIG** (`joint_eig`, the default) is the information an observation at $x$ is expected to
give about the model and its coefficients together:

$$
a_{\mathrm{joint}}(x) = H\big[\bar p(\cdot \mid x)\big] - \sum_m w_m \, \frac{1}{K_m} \sum_k H\big[p(\cdot \mid x, m, \theta_{m,k})\big]
$$

**Model EIG** (`structure_eig`; the launch scripts and the ForceBench records call it
`model_eig`) is the information about which model in the pool is right:

$$
a_{\mathrm{model}}(x) = H\big[\bar p(\cdot \mid x)\big] - \sum_m w_m \, H\big[\bar p_m(\cdot \mid x)\big]
$$

**Parameter EIG** (`parameter_eig`) is the information about the coefficients given the model,
averaged over models:

$$
a_{\mathrm{param}}(x) = \sum_m w_m \, H\big[\bar p_m(\cdot \mid x)\big] - \sum_m w_m \, \frac{1}{K_m} \sum_k H\big[p(\cdot \mid x, m, \theta_{m,k})\big] = a_{\mathrm{joint}}(x) - a_{\mathrm{model}}(x)
$$

**EPIG** (`epig`, expected predictive information gain) is the information about outcomes at a
fixed set $T$ of target inputs. An outcome $Y_x$ at the design and an outcome $Y_{x_*}$ at a
target are independent given one particle (the same model and coefficients), but not under the
BMA, and EPIG averages their mutual information over the targets:

$$
a_{\mathrm{EPIG}}(x) = \frac{1}{|T|} \sum_{x_* \in T} I\big(Y_x ; Y_{x_*} \mid D\big) ,
\qquad
I\big(Y_x ; Y_{x_*} \mid D\big) = \sum_{y, y_*} \bar p(y, y_*) \log \frac{\bar p(y, y_*)}{\bar p(y) \, \bar p(y_*)} ,
$$

$$
\bar p(y, y_*) = \sum_m w_m \, \frac{1}{K_m} \sum_k p(y \mid x, m, \theta_{m,k}) \, p(y_* \mid x_*, m, \theta_{m,k}) .
$$

$T$ has 32 points, drawn once at the start of the run (uniformly on [0, 1] in the synthetic
count benchmark, from the benchmark's own input sampler in ChemBench) and saved in `config.json`
as `epig_targets` together with their sha256.

**Random** (`random`) draws one candidate design from a random stream of its own, so that the
other random numbers of the run are not affected.

Two more criteria exist for reference. **Point disagreement** is the evidence-weighted spread of
the models' predicted means, $\sum_m w_m \big(\mu_m(x) - \bar\mu(x)\big)^2$ with $\bar\mu =
\sum_m w_m \mu_m$; it ignores the shape of the predictive distributions. **Space filling** picks
the candidate design farthest from the observed designs after scaling each input by its declared
range. The EIG-type arms record joint, Model and Parameter EIG and the point disagreement of
every design they evaluate, so that the arms can be compared on the same quantities.

**Exact sums and Monte Carlo.** For counts out of a fixed number of trials the outcome has
finitely many values (51 in the synthetic count benchmark: 0 to 50), so the entropies above are
exact sums and no outcome is sampled (`count_inference.information_from_components`). The
posterior bank holds up to 32 (`dev`) or 128 (`paper`) particles per model, drawn without
replacement, and the same bank is used for every candidate design, so differences between
designs are not resampling noise. Joint, Model and Parameter EIG come from one call
(`count_inference.choose`) and are all recorded in `acquisition/tNN.json`. For EPIG with
bounded counts both outcomes are enumerated exactly; unbounded counts are enumerated up to a
cut-off with one recorded tail category, which gives the information of a slightly coarsened
outcome, and continuous outcomes use predictive quantile intervals
(`count_inference.choose_epig`).

For positive, real-valued and vector outcomes the criteria are Monte Carlo estimates. Each draw
$s$ picks a model by weight, a particle of that model and an outcome $y_s$ from it, and

$$
\hat a_{\mathrm{joint}}(x) = \frac{1}{S} \sum_{s=1}^{S} \Big[ \log p(y_s \mid x, m_s, \theta_s) - \log \bar p(y_s \mid x) \Big] .
$$

The Model part uses $\log \bar p_{m_s}(y_s \mid x) - \log \bar p(y_s \mid x)$ on the same draws,
and the Parameter part is the rest. The runners record the Monte Carlo standard error beside each
value (`positive_inference.joint_eig`, `vector_inference.joint_eig`).

**Continuous design search.** With `--design-search ascent` the design is not limited to a finite
candidate set. ChemBench runs projected SPSA (simultaneous perturbation stochastic approximation)
on a unit box that maps onto the declared input ranges, from five starting points, for 20 to 500
steps (`design_search.projected_spsa`), and compares the end points with the same argmax rule.
The search keeps only particles whose predicted mean is finite and positive at a design and
records how many it dropped; if no design is valid, the round falls back once to the design a
random-arm run with the same seed would observe in that round, and the acquisition record says
so (`acquisition_status`). ForceBench climbs its criterion by gradient ascent over its two
design parameters, and the BoxingGym runner at its paper tag has its own version.
`--design-search pool`, the default, takes the argmax over the finite candidate set; ChemBench
refuses it for joint EIG and EPIG when a task has more than two inputs, which is true of every
ChemBench task, so its EIG-type runs use `ascent`.

The cell below prints the design policies the synthetic count runner accepts
(`--design-policy`) and the column of the criteria table each one maximizes, and the two EIG
parts the NeuronBench runner accepts.

In [2]:
from bayes_model_discovery.experimental import design_policies, vector_inference

print("synthetic count runner, --design-policy:")
for name in design_policies.POLICIES:
    default = "   (default)" if name == design_policies.DEFAULT_POLICY else ""
    print(f"  {name:20} maximizes {design_policies.OBJECTIVE_COLUMN[name]!r}{default}")
print("\nNeuronBench runner, --design-policy on an EIG cell:", ", ".join(vector_inference.EIG_PART_OBJECTIVES))

synthetic count runner, --design-policy:
  joint_eig            maximizes 'eig'   (default)
  structure_eig        maximizes 'structure_eig'
  point_disagreement   maximizes 'point_disagreement'
  uniform_random       maximizes 'uniform_random'
  epig                 maximizes 'epig'
  space_filling        maximizes 'min_normalized_distance'

NeuronBench runner, --design-policy on an EIG cell: structure_eig, parameter_eig


## 11. What never reaches the model

The held-out evaluation data are used only for scoring. They are not passed to fitting, design
selection or the proposer, and `config.json` states this as `evaluation_use`
(`scoring_only_never_fit_choose_or_propose` in the synthetic count runner).

The belief packet is built without any held-out input, true value or score:
`belief_packet.build_belief_packet` has no argument through which they could arrive, and a packet
does not change when only the evaluation set changes. Its **interface block** is the only
task-specific part. It is generated from the task specification and lists the input names and
ranges, derived inputs, the outcome type and support, the number of trials (for counts) and the
link function, with no free text, no world name and no true parameter value. With name masking
(`--mask-input-names`) even the input names are replaced by `X1`, `X2`, …, and the engine maps
them back internally.

## 12. The protocol of the paper runs

The paper's experiments on ChemBench, BoxingGym, ForceBench and NeuronBench follow one shared
procedure, the unified protocol (`u3` is its revision). The synthetic count benchmark is not part
of the paper; it serves development and tests.
[`USAGE.md`](../USAGE.md#reproducing-the-paper-runs) explains how a run is tied to the code that
produced it.

| | ChemBench | BoxingGym | ForceBench | NeuronBench |
| --- | --- | --- | --- | --- |
| Rounds after the initial design | 30 | 30 | 16 | 5 |
| Starting pool | initial batch of 12 proposals | initial batch of 12 proposals | initial batch of 12 proposals | four fixed mechanisms |
| Pool limit | 12 | 12 | 12 | 6 |
| Candidates per LLM call | 3 | 3 | 3 | 3 |
| Archive review rounds | 5, 10, 15, 20, 25, 30 | 5, 10, 15, 20, 25, 30 | 5, 10, 15 | none |
| Arms | joint EIG, Model EIG, Parameter EIG, random | all five | all five | all five |
| Seeds | 31, 37, 41 | 31, 37, 41 (peregrines: 31, 41, 47) | 31, 37, 41 | 31, 37, 41 |

The five arms are joint EIG, Model EIG, Parameter EIG, EPIG and random. Joint EIG is the primary
arm, and the other arms are reported beside it. ChemBench is not enabled in this release (section
13), but its runs are part of the paper.

### Round 0: the initial batch

1. Twelve independent requests are sent in parallel (`--initial-proposals 12`), each asking for
   one candidate. All requests of this first wave see the same packet.
2. Each returned candidate is validated, probed and fitted like any other candidate.
3. If fewer than twelve distinct candidates are admitted (because of duplicates, rejections or
   failed fits), up to two more waves follow. They ask only for the number still missing, and
   their packets show the candidates admitted so far. A gap that remains after the last wave is
   recorded as the shortfall.
4. The pool keeps at most twelve models (`--active-maximum 12`). ChemBench switches off its
   fixed catalogue of starting models (`--initial-catalogue off`), and ForceBench uses no
   catalogue pool and hides its law catalogue from the packet.

### Every round: one call, three candidates

Each round makes one LLM request, which asks for three candidates (`--proposal-limit 3`). The
request is the system prompt followed by the **multi-candidate paragraph, version 2**. In plain
words, the paragraph asks for

- a JSON array of exactly three candidate objects, each with exactly the keys the system prompt
  specifies;
- three distinct, non-equivalent hypotheses;
- at least two of them differing in their primary functional composition, not merely in
  parameterization or by adding or removing a correction term;
- at most one nested or incremental refinement of a model with high posterior weight, and only
  when the diagnostics in the packet support it.

What the engine does with the reply:

- The diversity requirement is an instruction only; the engine does not check it.
- Exact duplicates (the model id of a pool member, of an archive entry, or of an earlier
  candidate in the same reply) are discarded without fitting and never replaced, so at most three
  new models are fitted per round.
- Each remaining candidate is fitted from its prior on all public data. One ranking by log
  evidence then keeps the top twelve of the old and new models.
- A reply that breaks the format, or whose candidates all fail validation, gets one repair turn:
  the error is sent back once.

### Archive review rounds

Every fifth round is an archive review round: rounds 5, 10, …, 30 in the 30-round ChemBench and
BoxingGym runs, and rounds 5, 10 and 15 in the 16-round ForceBench runs. NeuronBench has none.

On these rounds the request carries a second paragraph after the multi-candidate paragraph, the
**archive review paragraph, version 1** (`--archive-review-rounds`). It tells the LLM that,
besides its three new candidates, it may ask for up to two archived models to be fitted again:
only models whose archive outcome is `retired_by_evidence` and whose form it now judges likely,
given that they were retired on less data. It may also ask for none. The reply is one JSON
object, `{"candidates": [...], "revive": [...]}`, where `revive` lists model ids exactly as the
archive gives them.

What the engine does with the revive list:

1. Only the first two ids count; later ones are refused as `over_limit`. An id is also refused if
   it names no archived model (`unknown`), a model in the pool (`active`), a model whose latest
   outcome is not `retired_by_evidence` (`not_retired`), or an id already listed (`repeated`).
   Refused ids are recorded and ignored; they cost no repair turn and no extra call.
2. If the reply has no revive list, nothing is revived, and the record says so
   (`revive_missing`).
3. Each valid model is refit from its prior on all current public data.
4. One ranking over the pool, the revived models and the round's new candidates keeps the top
   twelve. A revived model that makes the cut becomes `active` again in the archive; one that
   does not stays `retired_by_evidence`.
5. Everything is written to `revival/tNN.json`: the requested and valid ids, the refused ids with
   their reasons, the refit results with their log evidence, the ids that re-entered or were
   retired again, the evidence cutoff, the pool afterwards, and the wall time.

An archive review round adds no LLM call, because the revive list comes in the same reply as the
candidates. An older feature of the synthetic count runner, in which the engine itself refits
every retired model on fixed rounds (`--archive-reevaluation-rounds`, records in
`resurrection/tNN.json`), is not used by the protocol.

### What the LLM is given

- **System prompt.** ChemBench and BoxingGym send the file
  `tools/fb1_launch_c/proposer_system_v2_9_FA-C.txt` (version tag `v2.9.FA-C`) with
  `--prompt-variant`. ForceBench sends the same file or, for the oscillator world, a variant in
  the same folder whose operator list adds sine and cosine, together with a grammar that accepts
  them. NeuronBench sends its own mechanism prompt,
  `src/bayes_model_discovery/experimental/prompts/proposer_system_v3_0_mechanism.txt`, followed
  by the same multi-candidate paragraph.
- **Packet.** The belief packet of each round, with the archive in full. Input names are masked
  as `X1`, `X2`, …, and the packet contains no task description or world name; the task appears
  only through the interface block.
- **Model and decoding.** The launch scripts call `claude-opus-5-5` with
  `--llm-reasoning effort=medium,thinking=adaptive,display=summarized` and
  `--llm-max-tokens 16000`, and cap the provider calls of each run with `--max-paid-calls`
  (`FB_MAX_PAID_CALLS` in ForceBench).

### Identifiers

Every run records the digests of the texts it sent, so a run can be checked against the values
below. The cell computes them from the files and constants in this checkout: the packaged prompt
files, the paper's system prompt and its variant, and the two paragraphs, whose digests are taken
over the constants' bytes with the `{limit}` placeholder unfilled. Request records also carry the
digest of the composed system text (`composed_prompt_sha256`), which changes between rounds only
through the paragraphs. [`USAGE.md`](../USAGE.md#prompt-files-are-identifiers) lists the same
values; none of these texts is ever edited, and a new version is a new file.

In [3]:
from bayes_model_discovery.experimental import count_proposer


def sha256(data):
    return hashlib.sha256(data).hexdigest()


print("packaged prompt files (src/bayes_model_discovery/experimental/prompts/):")
for path in sorted(count_proposer.PROMPT_DIRECTORY.iterdir()):
    if path.is_file():
        print(f"  {path.name:38} {sha256(path.read_bytes())}")
print(f"default prompt version: {count_proposer.PROMPT_VERSION}")

print("\nsystem prompt of the paper runs (tools/fb1_launch_c/):")
for path in sorted(Path("tools/fb1_launch_c").glob("proposer_system_*.txt")):
    data = path.read_bytes()
    print(f"  {path.name:38} {sha256(data)}  {len(data):,} bytes, first line {data.decode().splitlines()[0]!r}")

print("\nparagraphs appended to the system prompt (count_proposer.py):")
for name in ("COMPOSITION_PARAGRAPH_V2", "ARCHIVE_REVIEW_PARAGRAPH"):
    data = getattr(count_proposer, name).encode()
    print(f"  {name:38} {sha256(data)}  {len(data)} bytes")

packaged prompt files (src/bayes_model_discovery/experimental/prompts/):
  pool_generator_v1.txt                  a4466c0f6a54e0380f630481f6501802e5a75a0e981d290f310a64834eecc1a4
  proposer_system_v2_0.txt               f095edf1adb3b6533a9e68e73f98ca1dee3f23cd077e424b092393cb32b93aef
  proposer_system_v2_1.txt               2edb4e0c11ba00cb6f07be4ddef48d4da75893a717c3c22ca767dfc2bd33a273
  proposer_system_v2_2.txt               e828e6ae587131766513d7bd7127ec2bcf302bcf02528797d2ec3327e6ab60ba
  proposer_system_v2_3.txt               1a5f9ba920a9bb5ded76206e3f7e3af2d86471fbde6e13345c752f9257493900
  proposer_system_v2_4.txt               1e0923f5a3bf9d76f0ecd2418cb5f2c5d1bca24fc49d91b5cf029279e8007461
  proposer_system_v2_4_1.txt             84229e9949c5ca5049f2afa5ec536966b6e24c60d02388f87da00d9e0f6596cb
  proposer_system_v2_5.txt               3969092f63f5ec6e76ae98ab9d44b8be62873f1dcdfd31dff80c5534a1e742ad
  proposer_system_v2_6_1.txt             ba5faab3136111b9523e9ac4b01bc6b3a26640

### Design choice in each arm

- **ChemBench.** The three EIG arms run with `--cell 101 --design-search ascent`, each climbing
  its own criterion (`--design-policy structure_eig` for Model EIG, `parameter_eig` for
  Parameter EIG, the default for joint EIG). The random arm is `--cell 100`.
- **BoxingGym.** The four active arms climb their
  own criterion on the task's declared input box, with the best design of the candidate set as a
  floor. Survival, whose candidate set is its whole legal design set, is searched exhaustively.
- **ForceBench.** Joint, Model and Parameter EIG use condition code 111 and EPIG uses 112, all
  with gradient ascent; the random arm (110) draws its designs at random.
- **NeuronBench.** Joint EIG is `--cell 111`, Model EIG `--cell 111 --design-policy
  structure_eig`, Parameter EIG `--cell 111 --design-policy parameter_eig`, EPIG `--cell 112`,
  random `--cell 110`.

Within one benchmark and world, every arm uses the same prompt, grammar, likelihood families,
priors, compute profile, initial-design rule and seeds; only the design criterion differs. The
EIG-type arms record joint, Model and Parameter EIG and the point disagreement of every design
they evaluate; in ForceBench and NeuronBench (`--record-design-criteria`) the random arm records
them too.

### Benchmark-specific settings

- **ChemBench.** Profile `paper`, `--noise 0.01`, initial design `fixed3`, `--fit-timeout 600`,
  a stop file, and tasks from `manifests/chembench_paper_manifest.json` (the medium-difficulty
  expansion tasks ran from a manifest kept in the private development tree).
- **BoxingGym.** Five of the ten tasks: death, dugongs, peregrines, hyperbolic and survival.
  Profile `paper`, initial design `random2` (two random designs), and
  `--d0-zero-count-unit half_count`, a correction that acts only when every initial response is
  zero. For peregrines, seed 37's two random initial designs both observe zero and the run
  cannot proceed, so peregrines uses seed 47 in place of 37.
- **ForceBench.** Five worlds (gravity, yukawa, coulomb_easy, oscillator, extra_dimensions), two
  random starting designs, the simulator's own integrator for coulomb_easy, oscillator and
  extra_dimensions, and the runner's earlier integrator for gravity and yukawa.
- **NeuronBench**, the exception. The candidates come from a fixed vocabulary of ten mechanism
  names. The run starts from four fixed mechanisms instead of an initial batch, keeps at most six
  models (profile `paper_neuron`), and has no archive review. Of the protocol's proposal rules
  it adopts only the three-candidate call with the multi-candidate paragraph. Six worlds, five
  rounds.

### What is recorded

- **Per call:** the packet; the digests of the system prompt, of the multi-candidate paragraph
  and, on review rounds, of the archive review paragraph; the digest of the composed prompt; the
  raw reply; validation and repair; which candidates were duplicates.
- **Per round:** the number of admitted candidates (at most three), the models retired and their
  gap to the cutoff, and on review rounds the revival record.
- **Per design:** joint, Model and Parameter EIG and the point disagreement of every evaluated
  design.

## 13. The benchmarks

Five benchmarks, each with its own runner. The README lists their external code and licenses,
and [`USAGE.md`](../USAGE.md#run-one-cell-with-scriptsbmd_runpy) shows how to run one cell of
each with `scripts/bmd_run.py`. All runners write the run folder described in
[`FLOW.md`](../FLOW.md#the-run-folder); the differences are listed below.

### SynthCount: the synthetic count benchmark (`core_count`)

The engine generates the worlds itself (`core_count.CoreWorld`), so this benchmark needs nothing
beyond the base install; it is the quickest way to see the whole loop, and it is used for
development and tests rather than in the paper. The code calls it CORE (`suite` = `core` in
`points.csv`).

- **Input and outcome.** One input, a dose $u \in [0, 1]$ (with $z = 2u - 1$); the outcome is a
  count of successes out of 50 trials.
- **True law.** The success probability $p$ follows `--structure`:
  `in`, $\operatorname{logit} p = b_0 + b_1 z$ (one of the four starting structures, so the truth
  is in the starting pool); `out` (the default), $\operatorname{logit} p = b_0 + b_1 z + 0.8
  \tanh(3z)$; `den`, $\operatorname{logit} p = b_0 + b_1 z / (1.6 + z)$; `pow`,
  $\operatorname{logit} p = b_0 + b_1 (u + 0.25)^{2.4}$.
- **Worlds.** $(b_0, b_1)$ comes from a table selected by `--seed`: 11 → (−0.8, 1.4),
  17 → (−0.2, 1.0) and 23 → (0.5, 0.8) are the research worlds; 31, 37, 41 and 99 are
  development worlds. `--seed` accepts only these values; `--replicate` gives independent
  repetitions within a world.
- **Noise.** `--truth-family binomial`, `beta_binomial` (the default: $p$ varies between
  observations as a beta distribution with concentration 9) or `zi_binomial` (each outcome is
  replaced by zero with probability 0.2).
- **Designs and scores.** The initial design is $u$ = 0.1, 0.35, 0.65 and 0.9; candidate designs
  are an evenly spaced grid on [0, 1] (16 points in `dev`, 100 in `paper`). Every round is scored
  by NLL, 90% interval coverage and width, KL and the RMSE of the predicted mean on 32 (`dev`) or
  256 (`paper`) held-out inputs with four outcomes each.
- **Normal-outcome worlds.** `--structure gauss_den`, `gauss_pow` or `gauss_call` with
  `--truth-family normal` and a condition code whose middle digit is 0 (for example `--cell 101`)
  give a real-valued outcome with mean $b_0 + b_1 u / (0.07 + u)$, $b_0 + b_1 u^{0.40}$ or
  $b_0 + b_1 \tanh(5z)$. The true noise standard deviation is 0.05 times the standard deviation of
  the mean over a 100-point grid of [0, 1]; candidates infer their own noise scale, and the true
  value never enters fitting.
- **Limitations.** The mock proposer answers from fixed templates, so a mock run tests the
  machinery, not the quality of LLM proposals. The runner's default profile is `paper`, which is
  much slower than `dev`. It has no Parameter EIG arm, but every EIG-type run records joint,
  Model and Parameter EIG for each candidate design.

### ChemBench (`chembench_core`): not enabled in this release

ChemBench's true laws, input ranges and true values come from LLM-AutoSciLab, whose repository
states no license. The ChemBench code stays in the tree for the record, but the run script
refuses it, the provisioning script does not fetch the dependency, and its tests skip; it will be
enabled once the license is clarified. The paper's ChemBench runs used this code with the dependency installed.

- **Task.** Each task is a chemical rate law, mostly from enzyme kinetics, with seven named
  inputs, including temperature and pH. An experiment sets all seven inputs and measures the rate
  with noise (`--noise`, required; 0.01 in the paper runs). Candidates are expressions for the log
  of the mean rate with a positive likelihood family (`--family lognormal`, the default, or
  `gamma`); with name masking the LLM sees the inputs as `X1` … `X7`.
- **Tasks.** Rows of a task manifest (`--manifest`, `--task`):
  `manifests/chembench_dev_manifest.json` holds one development task;
  `manifests/chembench_paper_manifest.json` holds nine tasks, three reaction families
  (`c0_michaelis_menten`, `c2_product_inhibition`, `c5_pingpong_bisubstrate`) at three difficulty
  levels; the expansion runs used a manifest of 34 medium-difficulty tasks (rows 1–24 the primary
  selection, rows 25–34 reserves) that is kept in the private development tree.
- **Scores.** The official error, nMSLE (`nmse_mda`), is computed on the benchmark's official set
  of 1,000 inputs, drawn with seed 42 after the run finishes: the mean squared difference of
  log1p(predicted mean) and log1p(true rate), divided by the variance of log1p(true rate), where
  the truth is the noise-free first-order law (`primary_rate`); `rmsle` is the square root of the
  unnormalized mean. Interval scores (coverage90, width90) are computed against held-out
  observations (`heldout_observation`), and KL against the noise-free observation including the
  benchmark's temperature and pH factor (`noiseless_observation`), so `kl` and `nmse_mda`
  measure different things. The engine's own held-out set gives the same scores.
- **Run folder.** No `scores.jsonl`, `archive.json` or `evaluation_scoring_only.npz`: the runner
  scores its checkpoints after the run (`evaluation_inputs.npz`, `reference_after_finish.npz`,
  `scores_after_finish.json`, and the same three files with the prefix `official_` for the
  official set), and the archive is part of every packet. It also writes `pool_weights.jsonl`,
  `round_timings.jsonl`, `frozen_d0_policy.json` (the response scale computed from the initial
  observations and then kept fixed), `final_raw_predictions.npz`, `revival/tNN.json`, the
  receipts of how the initial pool was made, `candidate_failures.jsonl` and, with name masking,
  `masked_model_ids.jsonl`.
- **Limitations.** The paper runs use four arms; there is no EPIG arm. Every task has seven
  inputs, so the EIG-type runs need `--design-search ascent`. `points.csv` (`source_path`) and
  `config.json` (`adapter_source`, `oracle_package`, `prompt_variant`) record absolute paths of
  the machine that made the run. The runner's defaults are 40 rounds and profile `dev`; paper
  runs pass `--profile paper --rounds 30`.

### BoxingGym (`boxinggym_core`)

BoxingGym is a set of simulated experimental environments. The runner wraps ten of them behind
one interface (`boxinggym_core.TASKS`); each task declares its inputs, its outcome type and its
starting structures.

| Task | Outcome type |
| --- | --- |
| `death` | bounded count |
| `peregrines` | unbounded count |
| `dugongs` | positive value |
| `survival`, `hyperbolic`, `irt`, `moral_machines` | binary |
| `location`, `emotion` | real value |
| `lotka_volterra` | trajectory |

- **Scores.** The runner calls the environment's own `run_experiment`, `step` and
  `evaluate_predictions`, and reports two kinds of score on the same held-out points: the
  environment's own evaluation and the engine's predictive scores (NLL, coverage90, width90). It
  records KL for every task except peregrines, where `--kl-all-tasks` turns it on.
  `--q-policy exclude_observed` removes evaluation points that equal an observed design.
- **Seeds.** The seed also seeds the environment, so `--seed` fixes the environment's hidden
  parameters as well as the engine's random streams. For peregrines and dugongs, whose
  environments draw with PyMC and would otherwise reseed from the operating system on every call,
  the runner injects seeds derived from `--seed` for the duration of the run
  (`env_seed_injection` in `config.json`).
- **External code.** BoxingGym, https://github.com/kanishkg/boxing-gym, at commit
  `b43e38cb03d09c13efa9cf4d9bae740d51157bfd` (MIT); the package used is `boxing_gym.envs` in the
  checkout's `src/` folder. The paper runs used a fork at `06c790aa8e23c1f8f1858f0955ec4ffc9e4c61ba`
  that changes four files under `src/boxing_gym/agents/`; `src/boxing_gym/envs/` is identical,
  and the only thing the environments import from `agents` is `construct_dataframe`, which the
  fork leaves unchanged. The tests accept only the upstream commit.
- **Run folder.** The core files, plus `points_mda.csv` (the score rows in the format of a
  published comparison method) and `frozen_d0_policy.json` (the response center and scale
  computed from the initial observations and then kept fixed, so priors are not recalibrated).
- **The paper runs** used this runner with its protocol options switched on: runs of 30 rounds
  (the default is 20), archive review rounds, the Model and Parameter EIG arms with a choice of
  the number of EPIG target points, continuous design search for the four EIG-type arms, name
  masking, a zero-count correction of the initial design (`--d0-zero-count-unit half_count`)
  and a numerically stable form of $\log(1 - e^{x})$.
- **Limitations.** This tree's runner lacks the protocol options above. With `random2`, some
  seeds give an initial design whose responses are all zero; for peregrines this happens with
  seed 37, and the paper runs use seed 47 in its place. Upstream
  `DirectDeath.get_goal_eval_question` overwrites the question text and loses the time value; the
  runner never reads the question string, only the numeric evaluation points, so it is not
  affected.

### NeuronBench (`neuron_core`)

Each world is a simulated neuron from the `neuronbench` package: `z_rebound`, `h_sag`,
`na_fatigue`, `ca_rebound`, `d_type` and `textbook_M` (`--world`; `--list-worlds` prints them).
The task is to find which ion-channel mechanism the neuron has.

- **Experiments.** An experiment is a current-injection protocol: a sequence of time segments,
  each with an injected current. The runner chooses among a fixed set of protocols, and each
  protocol can be run once.
- **Observations.** The simulated voltage trace, summarized as a vector of features; the
  `paper_neuron` profile uses the benchmark's six evaluation features.
- **Candidates.** One of ten named Hodgkin–Huxley-style mechanisms: `leak`, `sodium`,
  `potassium`, `slow_na`, `h_current`, `m_current`, `inward`, `rebound_inward_slow`,
  `rebound_inward_fast` and `rebound_outward`. Each has one positive conductance multiplier and a
  Gaussian likelihood on the features.
- **Fitting.** SMC through a black-box forward model wrapped as a PyTensor operation
  (`vector_inference.py`); the LLM call goes through `neuron_proposer.py`. The worlds are loaded
  deterministically, so `--seed` drives only the engine's random streams.
- **Profiles.** `dev` (the default) and `paper` keep a pool of twelve; `paper_neuron`, the profile
  of the paper runs, keeps at most six.
- **Run folder.** `proposals.jsonl` and `acquisition.jsonl` instead of the `proposals/` and
  `acquisition/` folders (with `--record-design-criteria`, each acquisition line holds joint,
  Model and Parameter EIG and the point disagreement of every available design),
  `observations/dNN.npz` with the simulated trace and its features for each observed design, and
  no `packets/` folder.
- **Limitations.** The batch forward backend (`--forward-backend batch`) is a vectorized copy of
  the simulator; it needs an equivalence report made on the same machine and Python
  (`--forward-equivalence-report`) and numpy's AVX-512 code paths disabled
  (`NPY_DISABLE_CPU_FEATURES`). The default `native` backend needs none of this. NeuronBench
  departs from the protocol as described in section 12, and the LLM chooses among names rather
  than writing expressions.

### ForceBench (`tools/fb1_snapshot/run_gravity.py`)

The task is to find the force law that governs a simulated physical system from trajectories.
The runner is a standalone script outside the installed package, in its own jax environment; it
is configured through `FB1_*` environment variables, which the launch script
`tools/fb1_launch_u/run_cell.sh` sets.

- **Worlds.** `gravity`, `yukawa`, `coulomb_easy`, `oscillator` and `extra_dimensions`. The true
  trajectories come from the DiscoverPhysics simulator, in the `ScienceAgent` folder of its
  checkout.
- **Experiments.** An experiment chooses two body parameters $(p_1, p_2)$, each in [0.1, 10].
- **Candidates.** A force-law expression in the separation $r$, the parameters $p_1$ and $p_2$,
  and time $t$; with name masking the LLM sees them as `X1` … `X4`.
- **Prediction.** The runner integrates each candidate's trajectories itself, in jax
  (`fb1_forward.py`): with the simulator's own integration method and settings (the *matched*
  integrator) for `coulomb_easy`, `oscillator` and `extra_dimensions`, and with the runner's
  earlier integrator (*legacy*) for `gravity` and `yukawa`.
- **Scoring.** At fixed designs: the three headline pairs (1, 1), (2, 1) and (1, 2), and six
  shifted pairs with $p_1 \in \{3, 4, 5\}$ and $p_2 \in \{3, 5\}$.
- **Launch script.** `run_cell.sh <world> <seed> <joint_eig|model_eig|parameter_eig|epig|random>
  <new output folder> <paid|mock> [OPTS]`, where `OPTS` is a comma-separated list:
  `grammar_trig=1` (add sine and cosine to the grammar), `prompt=<file in tools/fb1_launch_c>`,
  `epig_targets=score_q` and `engine=legacy|matched`. The runner also accepts the protocol options
  as flags (`--initial-proposals`, `--active-maximum`, `--proposal-limit`,
  `--archive-review-rounds`, `--design-policy`, `--packet-archive`, `--catalogue`).
- **Run files.** Flat files that share one prefix: `<run>.json` (settings, per-round records,
  the proposer settings with the prompt file and its sha256, and `code_sha256`),
  `<run>_numerics.json`, `<run>_packets/`, `<run>_proposer/`, `<run>_revival/tNN.json`,
  `<run>_evaluation_only.npz`, saved predictive components for scoring and KL,
  `<run>_masked_model_ids.jsonl`, and `<run>_gate_failure.json`, written only when the
  integration-accuracy check (the same trajectories with half the step size) rejects every
  candidate design, which stops the run.
- **Limitations.** Environment-variable configuration and flat output files, unlike the other
  runners. Many comments and docstrings in `tools/fb1_snapshot/` are not in English. Memory use
  is high: several gigabytes per run, and about 11 GB for an EPIG run on `gravity`. The launch
  script writes the digest of the prompt file it sends into the run log but does not refuse a
  different file, so check `prompt_sha256` in the run file against the identifiers of section 12.

## 14. Glossary

Terms you will meet in the code, on the command line and in run folders, grouped by topic.
[`FLOW.md`](../FLOW.md#glossary) has a shorter version.

### The loop

**Round, `tNN`.** One pass of the loop. Round 0 observes the initial design and fits the starting
pool; every later round adds exactly one designed observation, refits the pool, asks for new
candidates and scores the result. Files named `tNN` belong to round NN (`packets/t03.json`).

**D0 (initial design).** The observations made before round 1, at designs fixed by a rule rather
than chosen by a criterion (`--d0-policy`; section 4).

**Belief packet.** Everything the LLM receives besides the system prompt, built from public data
only: the observations, the pool with evidence weights and log evidence, posterior summaries of
each model's coefficients, the best model's standardized residuals, the archive of earlier
proposals, the constraints of the run and the interface block. Most runners save each packet as
`packets/tNN.json`, and its sha256 appears in the request record.

**Interface block.** The only task-specific part of a packet, generated from the task
specification: input names and ranges, derived inputs, the outcome type, its support, the number
of trials (for counts) and the link function. No free text, no world names, no true values.

**Candidate (proposal).** One hypothesis returned by the LLM: an expression for the mean on the
link scale, a likelihood family and the role of each coefficient, written as JSON. In NeuronBench
a candidate is a mechanism name instead.

**Validation, repair turn.** The checks a candidate must pass before it is fitted (section 6). A
reply that fails is answered once with the error message (`--repair-attempts`).

**Model identity (`model_id`).** A canonical key computed from the expression's syntax tree, the
coefficient roles and the likelihood family (schema `ast_roles_family_ac_v2`); section 6 lists
what changes it.

**Duplicate.** A candidate whose model id equals one already in the pool, in the archive or
earlier in the same reply. It is recorded and not fitted again.

**Prior probe.** 64 coefficient draws from the candidate's fixed priors, evaluated on a 16-point
grid over the declared input range; a non-finite value rejects the candidate without a fit. The
same draws give the `prior_predictive` summary saved in `fits/*.json`.

**SMC (sequential Monte Carlo).** The sampler that fits each model (`pm.sample_smc` in PyMC); it
moves particles from the prior to the posterior through tempered distributions and estimates the
marginal likelihood on the way. Every fit starts from the prior, uses all public data and runs
two chains.

**Particle.** One posterior draw of a model's coefficients. A fit keeps a fixed number of
particles per chain, set by the profile.

**Evidence (marginal likelihood, `logZ`, `log_evidence`).** $p(D \mid m)$, the probability of the
data under model $m$ with its coefficients integrated over their prior. SMC estimates it per chain
(`per_chain_log_evidence`), and the chains are averaged on the evidence scale.

**Evidence weight (model weight).** $w_m = Z_m / \sum_j Z_j$ over the models in the pool, which is
a uniform prior over the distinct models in the pool; written as `model_weights` in
`scores.jsonl` (ChemBench: `pool_weights.jsonl`). Section 8 explains why it is conditional on the
pool.

**One-step log score.** Before a pool member is refit on a new observation, its previous fit
gives the log predictive probability of that observation. The packet shows each model's history
of these scores.

**Pool (active pool, `active_maximum`).** The models currently kept: at most twelve (six in the
NeuronBench paper profile).

**Admission.** A candidate is admitted when it passes validation and the prior probe, its fit
succeeds, and its evidence ranks within the pool limit.

**Retired by evidence (`retired_by_evidence`).** A model that dropped out of the pool because
other models had higher evidence. Its archive entry records the log evidence at retirement, the
gap to the cutoff in nats, the rounds it was active and the number of observations at the time.

**Archive (`archive.json`).** The history of every model proposed or held in a run, each with an
outcome (`active`, `retired_by_evidence`, `rejected`, `duplicate`, `fit_failed`) and a structured
reason. The LLM sees it in every packet and is asked not to propose those models again. ChemBench
keeps it only inside the packets.

**Archive review round (`--archive-review-rounds`).** A round on which the request carries an
extra paragraph inviting the LLM to name up to two archived models with outcome
`retired_by_evidence` to be fitted again, besides its new candidates. Every fifth round in the
paper protocol, except in NeuronBench.

**Revive.** The LLM's request, on an archive review round, to refit a retired model. A valid
request is refit from the prior on all current public data and competes in the same ranking as
the round's new candidates; other ids are recorded with the reason they were refused. Records:
`revival/tNN.json`.

**Resurrection (`--archive-reevaluation-rounds`).** An older, different feature of the synthetic
count runner: on the listed rounds the engine itself refits every evidence-retired model.
Records: `resurrection/tNN.json`. The paper protocol does not use it.

**Burst (initial proposal batch).** At round 0, `--initial-proposals K` independent requests for
one candidate each, sent in parallel; at most two top-up waves follow, each asking only for the
number still missing (the shortfall). Records: `proposals/t00_burst.json`,
`packets/t00_burst_w<wave>.json`.

**K, k and K_m.** Three different numbers. K is the size of the initial batch
(`--initial-proposals`); k is the number of candidates asked for in one request
(`--proposal-limit`, 1 to 3); $K_m$ is the number of posterior particles of model $m$ used in a
calculation.

**Composition paragraph (multi-candidate paragraph).** Text appended to the system prompt when a
request asks for more than one candidate. Version `v1` is the default; version `v2` is the one
the paper runs used (section 12).

**Proposal policy (`--proposal-policy`, synthetic count runner).** `P1`, the default, makes one
proposal request on every round, asking for `--proposal-limit` candidates (default 1). `P4` asks
only at rounds 5, 10, 15 and 20, for three candidates each time, and is kept as a legacy
comparison.

**Scoring phase.** The synthetic count runner scores the pool twice per round, before and after
the round's proposals (`before_proposal`, `after_proposal`; the first row is `initial`). When the
proposals did not change the pool, the second row is copied and marked `post_equals_pre: true`.

### Design criteria

**Design.** The inputs of one experiment, for example the dose $u$ in the synthetic count
benchmark. The criterion is evaluated on candidate designs: a fixed grid or random sample (the
"design pool"), or points found by a continuous search.

**Arm.** The design criterion a run uses. The paper compares five arms: `joint_eig`, `model_eig`,
`parameter_eig`, `epig` and `random`.

**Joint EIG (`joint_eig`).** The expected information gain about the model and its coefficients
together, $I((M, \theta); Y \mid x, D)$. The default criterion.

**Model EIG (`model_eig`, `structure_eig`).** The expected information gain about which model in
the pool is right, $I(M; Y \mid x, D)$. `--design-policy structure_eig` selects it; records call
it `structure_eig`, except the ForceBench records, which call it `model_eig`.

**Parameter EIG (`parameter_eig`).** The expected information gain about the coefficients given
the model, averaged over models; joint EIG minus Model EIG.

**EPIG (expected predictive information gain).** The expected information an observation at $x$
gives about outcomes at a fixed set of target inputs, averaged over the targets. The 32 targets
are drawn once at the start of a run and saved as `epig_targets` in `config.json`.

**Random arm.** One candidate design drawn from a dedicated random stream, so the run's other
random numbers are not affected.

**Point disagreement.** How far apart the pool members' predicted means are at a design, weighted
by evidence; recorded beside the other criteria and available as a criterion of its own.

**Space filling.** A reference criterion that picks the candidate design farthest from the
designs already observed, after scaling each input by its declared range.

**Design search (`--design-search pool|ascent`).** `pool` takes the best design in a finite
candidate set; `ascent` searches the declared input box continuously and keeps the best of the
end points. Ties within $10^{-12}$ of the maximum go to the smallest design.

**Primary arm (headline).** Joint EIG is the primary arm of the paper protocol, and the other
arms are reported beside it. ForceBench also calls three of its scoring designs, (1, 1), (2, 1)
and (1, 2), the headline designs.

### Scores

**Held-out evaluation set (`q`).** Inputs and outcomes never used for fitting, choosing designs
or proposing; they only score predictions.

**BMA (Bayesian model averaging).** The pool's predictive distribution: each model's predictive
distribution, averaged over its particles, mixed with the evidence weights.

**NLL.** Negative log predictive probability (or density) of the held-out observations under the
BMA, in nats per observation. Lower is better.

**coverage90, width90.** The fraction of held-out observations inside the 90% predictive interval,
and the average width of that interval.

**KL (`kl`).** The KL divergence from the true distribution of the outcome to the BMA predictive,
averaged over the evaluation inputs.

**nMSLE (`nmse_mda`, `nmsle`).** ChemBench's primary error: the mean squared difference of
log1p(predicted mean) and log1p(true value) over the 1,000 official inputs, divided by the
variance of log1p(true value). `rmsle` is the square root of the unnormalized mean.

**Official score and engine score.** The official score is the benchmark's own evaluation
(ChemBench's error on its official set, or a BoxingGym environment's `evaluate_predictions`);
engine scores (NLL, coverage90, width90, KL) are computed by this repository from the BMA
predictive.

**Checkpoint.** A scored state of the pool.

**`mda` in names.** `nmse_mda`, `points_mda.csv` and some profile settings carry `mda` because
they were chosen so that results can be compared with a published model-discovery method that
the paper compares against.

### Runs and settings

**World.** One task instance with a fixed true law. Synthetic count: chosen by `--seed` with
`--structure` and `--truth-family`. ChemBench: `--task`, a row of a task manifest. BoxingGym:
`--task` and `--seed`. NeuronBench: `--world`, one of six. ForceBench: one of five world names.

**Seed.** What a seed controls differs by benchmark. In the synthetic count runner `--seed`
selects the world, and `--replicate` gives independent repetitions within it. In ChemBench the
task's law, its noise model and the official evaluation inputs do not depend on the seed; the
seed drives the engine's random streams, including a random initial design and the random arm.
In BoxingGym the seed also fixes the environment's hidden parameters. In NeuronBench the world is
deterministic and the seed drives the engine's random streams.

**Condition code (`--cell`).** Three digits S F A. S: 0 keeps the starting structures, 1 allows
new structures. F: 0 keeps the starting likelihood family, 1 allows other families. A: 0 random
designs, 1 an EIG-type criterion, 2 EPIG. Example: `111` = new structures, new families, joint
EIG. With `000` no candidates are proposed at all.

**Cell (run).** One run: one world, one seed and one arm, in its own output folder.

**Profile (`--profile`).** A compute preset (`dev`, `paper`, and `paper_neuron` in NeuronBench);
all profiles use the same priors, grammar, likelihoods and sampler (section 7).

**Mock proposer (`--proposer mock`).** The default proposer. It returns scripted candidates from
fixed templates instead of calling an LLM, so a run costs nothing and needs no key; `config.json`
says so (`mock_fixed_template_responses_NOT_LLM`).

**Frozen pool (`--initial-pool-file`).** An initial pool generated once per task and seed and
saved to a file; every arm then starts from the same file, and round 0 makes no LLM call.

**Catalogue.** Two meanings. In ChemBench, the fixed starting pool of four regression models
(`--initial-catalogue on|off`). In ForceBench, a list of reference force laws that can be shown
in the packet; off in the paper runs.

**Name masking (`--mask-input-names`).** Input names are replaced by `X1`, `X2`, … before the LLM
sees them, so the names cannot reveal the task.

**Run label (`--run-label development|production`).** `development` is the default and the label
of the paper runs; it does not mean "test only". `production` is accepted only with a paper
profile, an LLM proposer, a condition code that proposes new structures and a registration
manifest the run is verified against (`--manifest-ref`).

**Prompt version and prompt variant.** `--prompt-version v2.6.7.1` selects a numbered prompt file
from `prompts/`; `--prompt-variant <file>` (ChemBench and BoxingGym runners) sends any file and is
allowed only with the `development` label. The paper runs send
`tools/fb1_launch_c/proposer_system_v2_9_FA-C.txt` (version tag `v2.9.FA-C`); "FA-C" is only part
of that file's name.

**Stop file (`--stop-file`, ChemBench).** If this file appears, the run stops before the next
round and scores the checkpoints it has completed.

**Paid-call guard (`--max-paid-calls`).** A per-run cap on provider calls. The call that would
exceed it is refused, the run stops with exit status 4, and `summary.json` records the stop
reason `budget_exhausted`.

**Unified protocol.** The shared procedure of the paper runs (section 12); `u3` in the paper tags
is its revision.

**SynthCount ("CORE").** The synthetic count benchmark; the runner is `core_count.py`, and its
rows in `points.csv` have `suite` = `core`.

### Records and provenance

**`config.json`.** The full settings of a run: every option, package versions, seeds, the prompt
version and sha256, and the sha256 of each engine source file.

**`code_sha` and the source digests.** `code_sha` is a column of `points.csv`: the sha256 of the
runner's source file that produced the row. The digests of the other engine files are in
`config.json` (`source_files_sha256`, `source_sha256` or `experimental_source_sha256`, depending
on the runner; `code_sha256` in a ForceBench run file). Any edit to an engine file, even to a
comment, changes the digests of later runs.

**Request record (`llm_transport/`).** One JSON file per provider request and response: model,
decoding settings, token usage, prompt and packet digests, the raw response, the parse result and
the duration. Key-shaped strings are masked before writing.

**Receipt.** A record that binds the settings of a run to hashes before anything runs, so the
intended settings can be compared with the executed ones; for example `label_receipt` in
`config.json`, or ChemBench's `initial_pool_receipt.json`, which describes how the initial pool
was made.

**Manifest.** Four different files share the word. `MIGRATION_MANIFEST.json` lists the files
carried over from the archived prototype, with their hashes. A task manifest (`--manifest` in
ChemBench) lists the tasks a runner can use. A registration manifest (`--manifest-ref`) is what
the `production` label is verified against. `MANIFEST.json` in a published ChemBench run lists
the sha256 of every file that was published and of every file that was left out.

**Code identity.** The trees the paper runs were launched from are tagged in the private
development repository, not here. In this repository a run is tied to its code by the sha256 of
every engine source file and of the prompt file, recorded in the run's `config.json`; requests to
reproduce a paper run go to the authors.

### Benchmark-specific terms

**Mechanism (NeuronBench).** One of ten named ion-channel mechanisms with one conductance
multiplier (section 13).

**Forward backend (NeuronBench, `--forward-backend native|batch`).** `native` (the default)
calls the neuronbench simulator; `batch` is a vectorized copy of it and requires an equivalence
report made on the same machine and Python (`--forward-equivalence-report`).

**Matched and legacy integrators (ForceBench).** `matched` integrates trajectories with the
simulator's own method and settings; `legacy` is the runner's earlier integrator.

**Headline and shifted designs (ForceBench).** The fixed $(p_1, p_2)$ pairs ForceBench is scored
on: three headline pairs and six shifted pairs (section 13).

**Replay.** Re-running archived experiments to check bit-for-bit agreement, or recomputing
quantities offline from a finished run.

**Positive control.** Worlds where the right answer is known, run to check the pipeline.

**`heavy` and `replay` test markers.** `heavy` tests re-run SMC and compare against frozen
results; `replay` tests replay archived runs and need the archive on local disk.

## Where next

- [01 · Engine basics](01_engine_basics.ipynb): one mock run of the synthetic count benchmark,
  walked through round by round, with the fit, the weights and one EIG step reproduced by calling
  the engine.
- [02 · What the LLM sees](02_what_the_llm_sees.ipynb): one belief packet and the system text
  exactly as they are sent.
- [03 · EIG decomposition](03_eig_decomposition.ipynb): joint EIG = Model EIG + Parameter EIG,
  and EPIG, computed on a hand-made table and on a saved posterior.
- [04 · Reading a run folder](04_reading_a_run_folder.ipynb): the records of a run, one score
  reproduced by hand, two runs compared.
- [`FLOW.md`](../FLOW.md), [`USAGE.md`](../USAGE.md) and
  [`REQUIREMENTS.md`](../REQUIREMENTS.md).